# 03 · Rate of clinical progression by baseline SAA kinetics

SAA-positive participants only. Each kinetic parameter (Fmax, TTT, AUC; Project 237, 24-hour assay, mean of replicates) is split at its median: High vs Low (reference) for magnitude parameters, Fast vs Slow (reference) for time parameters. Linear mixed models of UPDRS III, Hoehn and Yahr and MoCA on time × kinetic group.

Phenoconversion during follow-up (PRIMDIAG = 1 or 5 at any visit) is handled two ways:

1. **adjust** – converters kept; `conv` and `time:conv` entered as covariates → `analysis/lmm_kinetics_conv_adj_bladj/`
2. **exclude** – converters removed entirely; medians recomputed in the non-converters (sensitivity analysis) → `analysis/lmm_kinetics_excl_conv_bladj/`

Requires `results_baseline/prodromal_saa_longitudinal.csv` and `prodromal_saa_baseline_core.csv` from notebook 01.

In [ ]:
# Works in Google Colab (mounts Drive) and locally. Set the project folder here,
# or through the PPMI_SAA_BASE environment variable.
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

BASE_DIR = os.environ.get('PPMI_SAA_BASE',
                          '/content/drive/MyDrive/PPMI_Projects/prodromal_SAA_groups')

## Configuration

In [ ]:
# ------------------------------- CONFIG -------------------------------------
# Output folder for each way of handling phenoconversion (kept separate so the
# two analyses never overwrite each other).
LMM_OUT_NAMES = {
    'adjust':  'lmm_kinetics_conv_adj_bladj',    # converters kept, `conv` is a covariate
    'exclude': 'lmm_kinetics_excl_conv_bladj',   # converters removed (sensitivity analysis)
}

# Which participants
LMM_SAA_STATUS = 'Positive'    # 'Positive', 'Negative', or None for both
LMM_KINETICS_SOURCE  = 'P237'  # 'P237' / 'P299' / 'P155' from the wide file,
                               # or 'selected' to use SAA_*_BL with the filter below
LMM_KINETICS_PROJECT = 237     # only used when LMM_KINETICS_SOURCE == 'selected'
LMM_AGG = 'mean'               # replicate aggregation of the P* columns
LMM_MIN_VISITS = 2             # a participant needs this many outcome values

# Phenoconversion during follow-up, defined from PRIMDIAG
LMM_CONV_COL   = 'PRIMDIAG'
LMM_CONV_CODES = [1, 5]             # 1 = idiopathic PD, 5 = DLB
LMM_CONV_TIME_INTERACTION = True    # 'adjust' only: also add time:conv (adjusts the slope)
LMM_MEDIAN_AFTER_EXCLUSION = True   # 'exclude' only: True = median of non-converters,
                                    #                 False = full-cohort median

# Longitudinal outcomes: (column, display name, higher score means)
LMM_OUTCOMES = [
    ('updrs3_score', 'UPDRS III motor score', 'worse'),
    ('hy',           'Hoehn and Yahr stage',  'worse'),
    ('moca',         'MoCA score',            'better'),
    # ('pigd',       'PIGD score',            'worse'),
]

# Kinetic parameters, each split at its median.
# exposure_above=True  : exposure = above the median (magnitude parameters)
# exposure_above=False : exposure = at/below the median (time parameters; faster)
LMM_METRICS = {
    'Fmax': {'exposure': 'High', 'reference': 'Low',  'exposure_above': True},
    'TTT':  {'exposure': 'Fast', 'reference': 'Slow', 'exposure_above': False},
    'AUC':  {'exposure': 'High', 'reference': 'Low',  'exposure_above': True},
    # 'TimeToMaxSlope': {'exposure': 'Fast', 'reference': 'Slow', 'exposure_above': False},
    # 'MaxSlope':       {'exposure': 'High', 'reference': 'Low',  'exposure_above': True},
}

LMM_TIME_COL = 'years_from_BL_exact'
LMM_TIME_MAX = 12              # fit and plot only the first N years

# Fixed effects besides time and group. Any outcome NOT listed in
# LMM_OUTCOME_COVARIATES uses LMM_COVARIATES. A term can be a column name
# ('EDUCYRS') or a categorical term ('C(race)').
# In the 'adjust' analysis `conv` is inserted automatically (just before the
# outcome's own baseline term, bl_<outcome>), plus time:conv if requested above.
LMM_COVARIATES = ['age_bl', 'sex']
LMM_OUTCOME_COVARIATES = {
    'moca':         ['age_bl', 'sex', 'EDUCYRS', 'C(fampd_bin)', 'APOE_e4', 'bl_moca'],
    'hy':           ['age_bl', 'sex', 'C(fampd_bin)', 'bl_hy'],
    # 'pigd':       ['age_bl', 'sex', 'C(fampd_bin)', 'bl_pigd'],
    'updrs3_score': ['age_bl', 'sex', 'C(fampd_bin)', 'bl_updrs3_score'],
}
LMM_RANDOM_SLOPE = True        # random intercept + slope; False = intercept only

LMM_MT    = 'holm'             # correction across the parameters within an outcome
LMM_ALPHA = 0.05
LMM_CLOW, LMM_CHIGH = '#378ADD', '#D85A30'      # reference / exposure
LMM_DPI = 600

# Baseline value of every outcome is built as column bl_<outcome>. To adjust for it,
# add it to that outcome's list above ('time:bl_hy' would also adjust the slope).
# bl_* covariates are mean-centred inside each model. When an outcome's OWN baseline
# is in its list, the visit that supplied that value is removed from the response
# (ANCOVA-type LMM); otherwise that row would fit its own covariate exactly and the
# fit fails (LinAlgError).
LMM_BL_WINDOW = 0.5            # years: baseline = non-missing value closest to time 0 within this
LMM_BL_MIN_FOLLOWUP = 1        # post-baseline values needed when an outcome is baseline-adjusted

# Extra trajectory figure per kinetic parameter (all outcomes side by side), in
# addition to the grid figure: a list of names from LMM_METRICS, e.g. ['Fmax'];
# 'all' = one figure for every parameter; [] = none.
LMM_SEPARATE_FIGS = []
LMM_SEPARATE_SHARE_Y = False   # True = same y-axis across the outcome panels

LMM_YEAR_BIN = 'round'         # n-per-year tables: 'round' = nearest year, 'floor' = completed years
# ---------------------------- END OF CONFIG ---------------------------------

## Functions

In [ ]:
from pathlib import Path
import warnings
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')

LMM_BASE = Path(BASE_DIR)
LMM_SRC  = LMM_BASE / 'results_baseline' / 'prodromal_saa_longitudinal.csv'
LMM_WIDE = LMM_BASE / 'results_baseline' / 'prodromal_saa_baseline_core.csv'
if not LMM_SRC.exists():
    raise FileNotFoundError(f'Missing input: {LMM_SRC}  (run 01_build_dataset first)')

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': LMM_DPI, 'savefig.bbox': 'tight',
                     'font.size': 12, 'axes.titlesize': 12, 'axes.labelsize': 10,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.22, 'grid.linewidth': 0.5,
                     'pdf.fonttype': 42, 'ps.fonttype': 42})

LMM_ASSAY_NAME = {'P155': 'Project 155 (150-hour pilot assay)',
                  'P237': 'Project 237 (24-hour assay)',
                  'P299': 'Project 299 (35-hour assay)'}
LMM_MODES = ('adjust', 'exclude')


# ---------------------------------------------------------------------------
# Small helpers
# ---------------------------------------------------------------------------
def lmm_source_name():
    if LMM_KINETICS_SOURCE == 'selected':
        return f'Project {LMM_KINETICS_PROJECT} (pipeline-selected kinetics)'
    return LMM_ASSAY_NAME.get(LMM_KINETICS_SOURCE, LMM_KINETICS_SOURCE)


def lmm_p(p):
    if p is None or not np.isfinite(p):
        return ''
    return '<0.001' if p < 0.001 else f'{p:.3f}'


def lmm_ptxt(p):
    s = lmm_p(p)
    return 'P' + s if s.startswith('<') else 'P = ' + s


def lmm_stars(p):
    if p is None or not np.isfinite(p):
        return ''
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''


def lmm_fmt(est, lo, hi, dec=3):
    if not np.isfinite(est):
        return 'not estimable'
    return f'{est:.{dec}f} ({lo:.{dec}f} to {hi:.{dec}f})'


def lmm_covs_for(outcome, mode):
    """Covariate terms for one outcome. In 'adjust' mode `conv` is inserted just
    before the outcome's own baseline term (or appended if there is none)."""
    covs = list(LMM_OUTCOME_COVARIATES.get(outcome, LMM_COVARIATES))
    if mode == 'adjust' and 'conv' not in covs:
        bl = f'bl_{outcome}'
        if bl in covs:
            covs.insert(covs.index(bl), 'conv')
        else:
            covs.append('conv')
    return covs


def lmm_term_col(term):
    """Data column behind a covariate term: 'C(race)' -> 'race', 'EDUCYRS' -> 'EDUCYRS'."""
    term = term.strip()
    if term.startswith('time:'):                 # 'time:bl_hy' -> 'bl_hy'
        term = term[5:].strip()
    mt = re.fullmatch(r'\s*C\(\s*([A-Za-z_]\w*)\s*(,.*)?\)\s*', term)
    return mt.group(1) if mt else term.strip()


def lmm_terms_txt(outcome, mode):
    """Covariate part of the formula for one outcome, as printed."""
    covs = lmm_covs_for(outcome, mode)
    if mode == 'adjust' and LMM_CONV_TIME_INTERACTION and 'conv' in covs:
        covs = covs + ['time:conv']
    return ' + '.join(covs)


def lmm_bl_adj(outcome):
    """True if this outcome's own baseline (bl_<outcome>) is in its covariate list."""
    return f'bl_{outcome}' in lmm_covs_for(outcome, 'exclude')


# config checks -- stop here instead of silently ignoring a typo
lmm_bad_keys = [k for k in LMM_OUTCOME_COVARIATES if k not in [oc for oc, _n, _d in LMM_OUTCOMES]]
if lmm_bad_keys:
    raise KeyError(f'LMM_OUTCOME_COVARIATES has keys that are not outcomes: {lmm_bad_keys}')
lmm_sep_bad = [m for m in (LMM_SEPARATE_FIGS if isinstance(LMM_SEPARATE_FIGS, list) else [])
               if m not in LMM_METRICS]
if lmm_sep_bad:
    print(f'WARNING: LMM_SEPARATE_FIGS has unknown parameter(s): {lmm_sep_bad}; '
          f'choose from {list(LMM_METRICS)}')


# ---------------------------------------------------------------------------
# Number of participants at each follow-up year (from the data in each model)
# ---------------------------------------------------------------------------
def lmm_year_table(results, group_col, group_labels, bin_mode=LMM_YEAR_BIN):
    """results: list of (outcome display name, fitted result dict)."""
    blocks = []
    for oname, r in results:
        if '_data' not in r:
            continue
        d = r['_data'].copy()
        yr = np.round(d['time']) if bin_mode == 'round' else np.floor(d['time'])
        d['Year'] = yr.astype(int)
        t = (d.groupby(['Year', group_col]).PATNO.nunique()
             .unstack(fill_value=0).reindex(columns=[0.0, 1.0], fill_value=0))
        # last row = unique participants in the model (NOT the column sum)
        t.loc['Total (model n)'] = [d.loc[d[group_col] == 0, 'PATNO'].nunique(),
                                    d.loc[d[group_col] == 1, 'PATNO'].nunique()]
        t.columns = pd.MultiIndex.from_product([[oname], list(group_labels)])
        blocks.append(t)
    tab = pd.concat(blocks, axis=1).fillna(0).astype(int)
    tab.index.name = 'Year'
    return tab


def lmm_year_md(tab):
    cols = [f'{a}, {b}' for a, b in tab.columns]
    out = ['| Year | ' + ' | '.join(cols) + ' |', '|---' * (len(cols) + 1) + '|']
    for idx, row in tab.iterrows():
        out.append(f'| {idx} | ' + ' | '.join(str(v) for v in row.values) + ' |')
    return '\n'.join(out)


def lmm_n_per_year(run):
    """One table per kinetic parameter: participants per follow-up year and group."""
    for m, spec in LMM_METRICS.items():
        res = [(on, run['res'][(oc, m)]) for oc, on, _ in LMM_OUTCOMES if (oc, m) in run['res']]
        if not res:
            continue
        tab = lmm_year_table(res, 'exposure', (spec['reference'], spec['exposure']))
        print(f'\n=== {m}: {spec["exposure"]} vs {spec["reference"]} ===')
        print(tab.to_string())
        tab.to_csv(run['out'] / f'lmm_kinetics_n_per_year_{m}.csv')
        (run['out'] / f'lmm_kinetics_n_per_year_{m}.md').write_text(lmm_year_md(tab), encoding='utf-8')
    print('\nwritten to', run['out'])


# ---------------------------------------------------------------------------
# Main analysis
#   mode = 'adjust'  : converters kept, conv (+ time:conv) entered as covariates
#   mode = 'exclude' : converters removed entirely before any model is fitted
# ---------------------------------------------------------------------------
def run_lmm_kinetics(mode):
    if mode not in LMM_MODES:
        raise ValueError(f'mode must be one of {LMM_MODES}')
    adjust = mode == 'adjust'
    LMM_OUT = LMM_BASE / 'analysis' / LMM_OUT_NAMES[mode]
    LMM_OUT.mkdir(parents=True, exist_ok=True)

    # ---------------------------- data ----------------------------
    lmm_d = pd.read_csv(LMM_SRC, na_values=[''], low_memory=False).sort_values(['PATNO', 'visit_n'])
    lmm_d['time'] = pd.to_numeric(lmm_d[LMM_TIME_COL], errors='coerce')
    if LMM_TIME_MAX is not None:
        lmm_d = lmm_d[lmm_d.time <= LMM_TIME_MAX]

    lmm_bl = lmm_d[lmm_d.is_baseline == 1].set_index('PATNO')
    if LMM_SAA_STATUS is not None:
        lmm_bl = lmm_bl[lmm_bl.SAA_Status_BL == LMM_SAA_STATUS]

    # phenoconversion: PRIMDIAG in LMM_CONV_CODES at ANY visit within the
    # (already time-capped) follow-up window -> 1, otherwise 0
    if LMM_CONV_COL not in lmm_d.columns:
        raise KeyError(f'Column {LMM_CONV_COL} not found in {LMM_SRC.name}')
    lmm_pdx = pd.to_numeric(lmm_d[LMM_CONV_COL], errors='coerce')
    lmm_conv_ids = set(lmm_d.loc[lmm_pdx.isin(LMM_CONV_CODES), 'PATNO'])
    lmm_conv_flag = pd.Series(lmm_bl.index.isin(lmm_conv_ids).astype(float), index=lmm_bl.index)

    # baseline covariates, carried onto every visit row
    lmm_cov = pd.DataFrame({
        'age_bl': pd.to_numeric(lmm_bl.get('age_at_visit'), errors='coerce'),
        'sex': pd.to_numeric(lmm_bl.get('SEX'), errors='coerce'),
        'conv': lmm_conv_flag,
    })

    # kinetics, one assay only
    lmm_kin_cols = {}
    if LMM_KINETICS_SOURCE == 'selected':
        proj = pd.to_numeric(lmm_bl.SAA_Kinetics_Project_BL, errors='coerce')
        lmm_bl = lmm_bl[proj == LMM_KINETICS_PROJECT]
        for m in LMM_METRICS:
            lmm_kin_cols[m] = pd.to_numeric(lmm_bl.get(f'SAA_{m}_BL'), errors='coerce')
    else:
        if not LMM_WIDE.exists():
            raise FileNotFoundError(f'{LMM_WIDE} not found; it holds the P155_/P237_/P299_ '
                                    "columns. Set LMM_KINETICS_SOURCE = 'selected' instead.")
        lmm_w = pd.read_csv(LMM_WIDE, na_values=[''], low_memory=False).set_index('PATNO')
        for m in LMM_METRICS:
            c = f'{LMM_KINETICS_SOURCE}_{m}_{LMM_AGG}'
            if c not in lmm_w.columns:
                raise KeyError(f'{LMM_WIDE.name} has no column {c}. Check LMM_KINETICS_SOURCE/LMM_AGG.')
            lmm_kin_cols[m] = pd.to_numeric(lmm_w[c].reindex(lmm_bl.index), errors='coerce')

    lmm_kin = pd.DataFrame(lmm_kin_cols, index=lmm_bl.index)
    lmm_kin = lmm_kin[lmm_kin.notna().any(axis=1)]
    lmm_cov = lmm_cov.reindex(lmm_kin.index)
    lmm_n_before = len(lmm_kin)
    lmm_is_conv = lmm_cov['conv'] == 1
    lmm_excl_ids = [] if adjust else sorted(lmm_kin.index[lmm_is_conv])

    # Parameter-specific median split:
    # magnitude parameters: High (> median) vs Low (<= median; reference)
    # time parameters: Fast (<= median) vs Slow (> median; reference)
    # 'exclude' mode: median of the non-converters if LMM_MEDIAN_AFTER_EXCLUSION
    lmm_med_base = (lmm_kin[~lmm_is_conv] if (not adjust and LMM_MEDIAN_AFTER_EXCLUSION)
                    else lmm_kin)
    lmm_cuts = {}
    for m in LMM_METRICS:
        med = float(lmm_med_base[m].median()) if lmm_med_base[m].notna().any() else np.nan
        lmm_cuts[m] = med
        spec = LMM_METRICS[m]
        is_exposure = ((lmm_kin[m] > med) if spec['exposure_above']
                       else (lmm_kin[m] <= med))
        lmm_kin[f'{m}_exposure'] = np.where(
            lmm_kin[m].notna(), is_exposure.astype(float), np.nan)

    # converters per kinetic group (reference / exposure), counted before any exclusion
    lmm_conv_rows = []
    for m, spec in LMM_METRICS.items():
        g = pd.concat([lmm_kin[f'{m}_exposure'], lmm_cov['conv']], axis=1).dropna()
        for val, lab in ((0.0, spec['reference']), (1.0, spec['exposure'])):
            gg = g[g[f'{m}_exposure'] == val]
            row = {'parameter': m, 'group': lab, 'role': 'reference' if val == 0 else 'exposure'}
            if adjust:
                row.update({'n': len(gg), 'converters': int(gg.conv.sum())})
            else:
                row.update({'n_before': len(gg), 'excluded': int(gg.conv.sum()),
                            'n_after': int((gg.conv == 0).sum())})
            lmm_conv_rows.append(row)
    lmm_conv_counts = pd.DataFrame(lmm_conv_rows)
    if adjust:
        lmm_conv_counts.to_csv(LMM_OUT / 'lmm_kinetics_conversion_counts.csv', index=False)
    else:
        lmm_conv_counts.to_csv(LMM_OUT / 'lmm_kinetics_excluded_converters.csv', index=False)
        pd.Series(lmm_excl_ids, name='PATNO').to_csv(
            LMM_OUT / 'lmm_kinetics_excluded_converters_ids.csv', index=False)
        # EXCLUDE converters entirely (all visits, both kinetic groups)
        lmm_kin = lmm_kin[~lmm_is_conv]
        lmm_cov = lmm_cov[~lmm_is_conv].drop(columns='conv')

    lmm_long = lmm_d[lmm_d.PATNO.isin(lmm_kin.index)].copy()
    lmm_long = lmm_long.drop(columns=[c for c in lmm_cov.columns if c in lmm_long.columns])
    lmm_long = lmm_long.join(lmm_cov, on='PATNO').join(
        lmm_kin[[f'{m}_exposure' for m in LMM_METRICS]], on='PATNO')

    # baseline value of each outcome (bl_<outcome>): non-missing value closest to time 0
    LMM_BL_ROWS = {}
    for _oc, _on, _dr in LMM_OUTCOMES:
        if _oc not in lmm_long.columns:
            continue
        _v = lmm_long[['PATNO', 'time', _oc]].copy()
        _v[_oc] = pd.to_numeric(_v[_oc], errors='coerce')
        _v['_at'] = _v.time.abs()
        _v = _v[(_v._at <= LMM_BL_WINDOW) & _v[_oc].notna()].sort_values(['PATNO', '_at'])
        _first = _v.groupby('PATNO', sort=False).head(1)
        LMM_BL_ROWS[_oc] = set(_first.index)      # used only if the outcome is baseline-adjusted
        _blv = _first.set_index('PATNO')[_oc].rename(f'bl_{_oc}')
        lmm_long = lmm_long.drop(columns=[f'bl_{_oc}'], errors='ignore').join(_blv, on='PATNO')

    # every covariate column must exist -- stop here instead of silently dropping it
    lmm_missing = sorted({lmm_term_col(t) for oc, _n, _d in LMM_OUTCOMES
                          for t in lmm_covs_for(oc, mode)} - set(lmm_long.columns))
    if lmm_missing:
        raise KeyError(f'Covariate column(s) not found in the data: {lmm_missing}')

    print(f'Kinetics source: {lmm_source_name()}'
          + ('' if LMM_KINETICS_SOURCE == 'selected' else f', {LMM_AGG} of replicates'))
    status_txt = '' if LMM_SAA_STATUS is None else f' (SAA-{LMM_SAA_STATUS.lower()})'
    if adjust:
        print(f'Participants: {len(lmm_kin)}{status_txt}   visit rows: {len(lmm_long)}')
        print(f'conv = 1 if {LMM_CONV_COL} in {LMM_CONV_CODES} at any visit '
              f'(<= {LMM_TIME_MAX} y): {int(lmm_cov.conv.sum())} of {len(lmm_cov)} participants')
        print('Median cuts: ' + ', '.join(f'{m}={lmm_cuts[m]:.4g}' for m in LMM_METRICS))
    else:
        print(f'Converters excluded ({LMM_CONV_COL} in {LMM_CONV_CODES} at any visit, '
              f'<= {LMM_TIME_MAX} y): {len(lmm_excl_ids)} of {lmm_n_before} participants')
        print(f'Participants analysed (non-converters): {len(lmm_kin)}{status_txt}'
              f'   visit rows: {len(lmm_long)}')
        print(('Median cuts (non-converters): ' if LMM_MEDIAN_AFTER_EXCLUSION
               else 'Median cuts (full cohort): ')
              + ', '.join(f'{m}={lmm_cuts[m]:.4g}' for m in LMM_METRICS))
    print('Contrasts: ' + ', '.join(
        f"{m}={spec['exposure']} vs {spec['reference']} (reference)"
        for m, spec in LMM_METRICS.items()))
    print('Model: outcome ~ time * exposure + <covariates> + '
          + ('(1 + time | PATNO)' if LMM_RANDOM_SLOPE else '(1 | PATNO)')
          + '   [parameter-specific reference group]')
    for oc, oname, _d in LMM_OUTCOMES:
        print(f'   covariates, {oname:<22}: {lmm_terms_txt(oc, mode)}')
    print()
    print('Phenoconverters by kinetic group' if adjust else 'Converters excluded by kinetic group')
    print(lmm_conv_counts.to_string(index=False), '\n')

    # ---------------------------- fit ----------------------------
    def lmm_fit_one(outcome, metric):
        """Fit one LMM. Returns a dict of estimates, or None if it cannot be fitted."""
        exposure_col = f'{metric}_exposure'
        spec = LMM_METRICS[metric]
        covs_all = lmm_covs_for(outcome, mode)     # covariates for THIS outcome
        cols = ['PATNO', 'time', outcome, exposure_col] + [lmm_term_col(t) for t in covs_all]
        d = lmm_long[[c for c in dict.fromkeys(cols) if c in lmm_long.columns]].copy()
        d[outcome] = pd.to_numeric(d[outcome], errors='coerce')
        if lmm_bl_adj(outcome):                     # own baseline -> covariate, not response
            d = d[~d.index.isin(LMM_BL_ROWS.get(outcome, ()))]
        d = d.dropna()
        if d.empty:
            return None
        keep = d.groupby('PATNO')[outcome].size()
        d = d[d.PATNO.isin(keep[keep >= (LMM_BL_MIN_FOLLOWUP if lmm_bl_adj(outcome)
                                         else LMM_MIN_VISITS)].index)]
        if d.PATNO.nunique() < 10 or d[exposure_col].nunique() < 2:
            return None
        d = d.rename(columns={outcome: 'y', exposure_col: 'exposure'})

        if adjust:
            # covariates with no variation in this fit (e.g. no converters) are dropped
            covs = [t for t in covs_all if d[lmm_term_col(t)].nunique() > 1]
            dropped = [t for t in covs_all if t not in covs]
            terms = covs + (['time:conv'] if LMM_CONV_TIME_INTERACTION and 'conv' in covs else [])
            drop_note = f'dropped (no variation): {", ".join(dropped)}' if dropped else ''
        else:
            terms, drop_note = covs_all, ''

        for _c in [c for c in d.columns if c.startswith('bl_')]:   # centre baseline covariates
            d[_c] = d[_c] - d[_c].mean()
        formula = 'y ~ time * exposure' + (' + ' + ' + '.join(terms) if terms else '')
        note = ''
        res = None
        for re_formula in (['~time', None] if LMM_RANDOM_SLOPE else [None]):
            try:
                md = smf.mixedlm(formula, d, groups=d['PATNO'], re_formula=re_formula)
                r = md.fit(method='lbfgs', reml=True)
                if not np.all(np.isfinite(r.bse.values)):
                    raise ValueError('non-finite standard errors')
                res = r
                note = '' if re_formula else 'random intercept only (slope model did not converge)'
                break
            except Exception as e:
                note = f'{type(e).__name__}'
        note = '; '.join(x for x in (note, drop_note) if x)
        if res is None:
            return {'outcome': outcome, 'parameter': metric,
                    'exposure_label': spec['exposure'], 'reference_label': spec['reference'],
                    'contrast': f"{spec['exposure']} vs {spec['reference']}",
                    'note': f'not estimable ({note})',
                    'n_participants': int(d.PATNO.nunique()), 'n_observations': len(d),
                    'slope_reference': np.nan, 'slope_reference_lo': np.nan, 'slope_reference_hi': np.nan,
                    'slope_exposure': np.nan, 'slope_exposure_lo': np.nan, 'slope_exposure_hi': np.nan,
                    'diff': np.nan, 'diff_lo': np.nan, 'diff_hi': np.nan, 'p': np.nan}

        b, V = res.fe_params, res.cov_params().loc[res.fe_params.index, res.fe_params.index]
        names = list(b.index)
        it = 'time:exposure' if 'time:exposure' in names else 'exposure:time'

        def contrast(weights):
            w = np.zeros(len(names))
            for k, v in weights.items():
                w[names.index(k)] = v
            est = float(w @ b.values)
            se = float(np.sqrt(w @ V.values @ w))
            return est, est - 1.96 * se, est + 1.96 * se, se

        s_reference = contrast({'time': 1.0})
        s_exposure = contrast({'time': 1.0, it: 1.0})
        d_est, d_lo, d_hi, d_se = contrast({it: 1.0})
        pval = float(2 * stats.norm.sf(abs(d_est / d_se))) if d_se > 0 else np.nan

        return {'outcome': outcome, 'parameter': metric,
                'exposure_label': spec['exposure'], 'reference_label': spec['reference'],
                'contrast': f"{spec['exposure']} vs {spec['reference']}", 'note': note,
                'n_participants': int(d.PATNO.nunique()), 'n_observations': len(d),
                'n_reference': int((d.exposure == 0).sum()),
                'n_exposure': int((d.exposure == 1).sum()),
                'median_cut': lmm_cuts[metric],
                'slope_reference': s_reference[0], 'slope_reference_lo': s_reference[1],
                'slope_reference_hi': s_reference[2],
                'slope_exposure': s_exposure[0], 'slope_exposure_lo': s_exposure[1],
                'slope_exposure_hi': s_exposure[2],
                'diff': d_est, 'diff_lo': d_lo, 'diff_hi': d_hi, 'p': pval,
                'covariates': ' + '.join(terms),
                '_fe': b, '_names': names, '_it': it, '_data': d, '_covm': V.values,
                '_xbar': pd.Series(np.asarray(res.model.exog).mean(axis=0),
                                   index=res.model.exog_names)}

    lmm_res, lmm_rows = {}, []
    for oc, oname, _dir in LMM_OUTCOMES:
        if oc not in lmm_long.columns:
            print(f'  skipped {oc}: column not in the dataset')
            continue
        praw = []
        for m in LMM_METRICS:
            r = lmm_fit_one(oc, m)
            if r is None:
                continue
            lmm_res[(oc, m)] = r
            praw.append(r['p'])
        ps = np.array(praw, dtype=float)
        ok = np.isfinite(ps)
        padj = np.full_like(ps, np.nan)
        if ok.sum() and LMM_MT != 'none':
            padj[ok] = multipletests(ps[ok], method=LMM_MT)[1]
        i = 0
        for m in LMM_METRICS:
            if (oc, m) not in lmm_res:
                continue
            lmm_res[(oc, m)]['p_adj'] = float(padj[i]); i += 1
            lmm_rows.append({k: v for k, v in lmm_res[(oc, m)].items() if not k.startswith('_')})

    lmm_tab = pd.DataFrame(lmm_rows)
    lmm_tab.to_csv(LMM_OUT / 'lmm_kinetics_estimates.csv', index=False)

    # ---------------------------- markdown table ----------------------------
    lmm_md = ['| Outcome | Parameter | Contrast | n (reference/exposure obs) | '
              'Slope, reference (95% CI) | Slope, exposure (95% CI) | '
              'Difference exposure − reference (95% CI) | P | P adj. |',
              '|---|---|---|---|---|---|---|---|---|']
    for oc, oname, _dir in LMM_OUTCOMES:
        for m in LMM_METRICS:
            r = lmm_res.get((oc, m))
            if r is None:
                continue
            padj = lmm_p(r.get('p_adj'))
            if np.isfinite(r.get('p_adj', np.nan)) and r['p_adj'] < LMM_ALPHA:
                padj = f'**{padj}**'
            lmm_md.append(
                f"| {oname} | {m} | {r['contrast']} | "
                f"{r['n_participants']} ({r.get('n_reference','')}/{r.get('n_exposure','')}) | "
                f"{lmm_fmt(r['slope_reference'], r['slope_reference_lo'], r['slope_reference_hi'])} | "
                f"{lmm_fmt(r['slope_exposure'], r['slope_exposure_lo'], r['slope_exposure_hi'])} | "
                f"{lmm_fmt(r['diff'], r['diff_lo'], r['diff_hi'])} | "
                f"{lmm_p(r['p'])}{lmm_stars(r['p'])} | {padj} |")

    lmm_codes = " or ".join(str(c) for c in LMM_CONV_CODES)
    lmm_cov_txt = 'the outcome-specific covariates listed below'
    if adjust and any('conv' in lmm_covs_for(oc, mode) for oc, _n, _d in LMM_OUTCOMES):
        lmm_cov_txt += (', and phenoconversion during follow-up '
                        f'({LMM_CONV_COL} = {lmm_codes} at any visit'
                        + (', entered as a main effect and as an interaction with time)'
                           if LMM_CONV_TIME_INTERACTION else ', entered as a main effect)'))

    lmm_md += ['',
               'Linear mixed models of each longitudinal outcome on time (years from baseline), '
               'parameter-specific baseline kinetic group (median split), their '
               'interaction, and '
               + lmm_cov_txt
               + ', with '
               + ('a random intercept and random slope for time per participant.'
                  if LMM_RANDOM_SLOPE else 'a random intercept per participant.'),
               '']
    if not adjust:
        lmm_md += [f'Participants who phenoconverted during follow-up ({LMM_CONV_COL} = '
                   f'{lmm_codes} at any visit within '
                   f'{LMM_TIME_MAX} years; n = {len(lmm_excl_ids)}) were excluded from all analyses. '
                   + ('Median cut-offs were computed in the remaining participants.'
                      if LMM_MEDIAN_AFTER_EXCLUSION else
                      'Median cut-offs were taken from the full cohort before exclusion.'),
                   '']
    lmm_md += ['Slopes are annual rates of change in the units of each outcome. The difference '
               'column is the interaction term: the extra annual change in the exposure group '
               'relative to its parameter-specific reference group. Magnitude parameters use '
               'High vs Low (reference); time parameters use Fast vs Slow (reference). '
               'Confidence intervals are Wald intervals from the fixed-effect '
               'covariance matrix.',
               '',
               f'Kinetics are from {lmm_source_name()}. Kinetic parameters are assay-dependent '
               'and were not pooled across assays.',
               '',
               f'P adj. = P value corrected across the {len(LMM_METRICS)} kinetic parameters '
               f'within each outcome by the {LMM_MT.title()} procedure; values <{LMM_ALPHA} bold.']
    lmm_bl_used = [on for oc, on, _d in LMM_OUTCOMES if lmm_bl_adj(oc)]
    if lmm_bl_used:
        lmm_md += ['', f'Models for {", ".join(lmm_bl_used)} are adjusted for the baseline value of '
                   f'the same outcome (value closest to baseline within {LMM_BL_WINDOW} years, '
                   'mean-centred; terms as listed below). In these models the visit supplying the '
                   'baseline value was not used as a response (post-baseline visits only), and '
                   f'participants needed a baseline value and at least {LMM_BL_MIN_FOLLOWUP} '
                   'post-baseline value of the outcome.']
    lmm_md += ['', 'Covariates by outcome:'] + [f'- {oname}: {lmm_terms_txt(oc, mode)}'
                                                for oc, oname, _d in LMM_OUTCOMES]
    (LMM_OUT / 'lmm_kinetics_estimates.md').write_text('\n'.join(lmm_md), encoding='utf-8')

    # ---------------------------- FIGURE 1: trajectories grid ----------------------------
    def lmm_pred(r, exposure, tgrid):
        """Marginal fitted mean at the covariate means, with a 95% Wald band.
        Uses the fixed-effect covariance stored at fit time -- no refitting."""
        b, names, it = r['_fe'], r['_names'], r['_it']
        # covariates at their mean in the data of THIS model; for a categorical
        # term C(x) this is the mean of each dummy
        xbar = r['_xbar']
        est, lo, hi = [], [], []
        for t in tgrid:
            w = np.array([float(xbar.get(n, 0.0)) for n in names])
            w[names.index('Intercept')] = 1.0
            w[names.index('time')] = t
            if 'exposure' in names:
                w[names.index('exposure')] = exposure
            w[names.index(it)] = t * exposure
            for n in names:                     # time:X terms (conv, bl_*) at the mean of X
                if n.startswith('time:') and n != it and n[5:] in names:
                    w[names.index(n)] = t * float(xbar[n[5:]])
            e = float(w @ b.values)
            se = float(np.sqrt(w @ r['_covm'] @ w))
            est.append(e); lo.append(e - 1.96 * se); hi.append(e + 1.96 * se)
        return np.array(est), np.array(lo), np.array(hi)

    lmm_ocs = [(oc, on, dr) for oc, on, dr in LMM_OUTCOMES
               if any((oc, m) in lmm_res for m in LMM_METRICS)]
    lmm_tmax = float(LMM_TIME_MAX) if len(lmm_long) else 1.0
    lmm_tgrid = np.linspace(0, lmm_tmax, 40)

    fig, axes = plt.subplots(len(lmm_ocs), len(LMM_METRICS),
                             figsize=(3.15 * len(LMM_METRICS), 2.75 * len(lmm_ocs)),
                             squeeze=False)
    for i, (oc, oname, _dir) in enumerate(lmm_ocs):
        for j, m in enumerate(LMM_METRICS):
            ax = axes[i][j]
            r = lmm_res.get((oc, m))
            if r is None or '_fe' not in r:
                ax.text(0.5, 0.5, 'not estimable', ha='center', va='center',
                        transform=ax.transAxes, fontsize=10, color='#A32D2D')
                ax.set_xticks([]); ax.set_yticks([])
                continue
            for exposure, colr in ((0.0, LMM_CLOW), (1.0, LMM_CHIGH)):
                e, lo, hi = lmm_pred(r, exposure, lmm_tgrid)
                ax.plot(lmm_tgrid, e, color=colr, lw=1.6)
                ax.fill_between(lmm_tgrid, lo, hi, color=colr, alpha=0.13, linewidth=0)
            ax.set_xlim(0, lmm_tmax)
            ax.set_title(f'{m}: {r["contrast"]}\nΔslope {r["diff"]:+.3f}/y, '
                         f'{lmm_ptxt(r["p"])}{lmm_stars(r.get("p", np.nan))}', fontsize=12)
            if j == 0:
                ax.set_ylabel(oname, fontsize=12)
            if i == len(lmm_ocs) - 1:
                ax.set_xlabel('Years from baseline', fontsize=12)
    fig.legend(handles=[Line2D([0], [0], color=LMM_CLOW, lw=2, label='Reference group(Low/Slow)'),
                        Line2D([0], [0], color=LMM_CHIGH, lw=2, label='Exposure group(High/Fast)')],
               loc='upper center', bbox_to_anchor=(0.5, 0.985), ncol=2, frameon=False, fontsize=12)
    fig.suptitle(f'Model-estimated trajectories by baseline kinetic group — '
                 f'{lmm_source_name()}', fontsize=14, y=1.005)
    fig.tight_layout(rect=(0, 0, 1, 0.965))
    fig.savefig(LMM_OUT / 'fig_lmm_trajectories.png', dpi=LMM_DPI)
    fig.savefig(LMM_OUT / 'fig_lmm_trajectories.pdf')
    plt.close(fig)

    # ---------------------------- FIGURE 1b: one figure per selected parameter ----------------------------
    lmm_sep = (list(LMM_METRICS) if LMM_SEPARATE_FIGS == 'all'
               else [m for m in (LMM_SEPARATE_FIGS or []) if m in LMM_METRICS])
    for m in lmm_sep:
        spec = LMM_METRICS[m]
        fig, axes = plt.subplots(1, len(lmm_ocs), figsize=(3.4 * len(lmm_ocs), 3.5), squeeze=False)
        for j, (oc, oname, dr) in enumerate(lmm_ocs):
            ax = axes[0][j]
            r = lmm_res.get((oc, m))
            if r is None or '_fe' not in r:
                ax.text(0.5, 0.5, 'not estimable', ha='center', va='center',
                        transform=ax.transAxes, fontsize=11, color='#A32D2D')
                ax.set_xticks([]); ax.set_yticks([]); ax.set_title(oname, fontsize=12)
                continue
            for exposure, colr in ((0.0, LMM_CLOW), (1.0, LMM_CHIGH)):
                e, lo, hi = lmm_pred(r, exposure, lmm_tgrid)
                ax.plot(lmm_tgrid, e, color=colr, lw=1.8)
                ax.fill_between(lmm_tgrid, lo, hi, color=colr, alpha=0.13, linewidth=0)
            ax.set_xlim(0, lmm_tmax)
            dd = r['_data']
            n_ref = dd.loc[dd.exposure == 0, 'PATNO'].nunique()
            n_exp = dd.loc[dd.exposure == 1, 'PATNO'].nunique()
            ax.set_title(f'{oname}\nΔslope {r["diff"]:+.3f}/y, {lmm_ptxt(r["p"])}{lmm_stars(r["p"])}'
                         f'\nP adj. = {lmm_p(r.get("p_adj"))}   n = {n_ref}/{n_exp}', fontsize=11)
            ax.set_xlabel('Years from baseline', fontsize=11)
            ax.set_ylabel('Model-estimated score' if j == 0 else '', fontsize=11)
        if LMM_SEPARATE_SHARE_Y:
            lims = [a.get_ylim() for a in axes[0] if a.has_data()]
            if lims:
                for a in axes[0]:
                    a.set_ylim(min(l[0] for l in lims), max(l[1] for l in lims))
        fig.legend(handles=[Line2D([0], [0], color=LMM_CLOW, lw=2, label=f"{spec['reference']} (reference)"),
                            Line2D([0], [0], color=LMM_CHIGH, lw=2, label=spec['exposure'])],
                   loc='upper center', bbox_to_anchor=(0.5, 0.99), ncol=2, frameon=False, fontsize=12)
        fig.suptitle(f"{m}: {spec['exposure']} vs {spec['reference']} — {lmm_source_name()}",
                     fontsize=13, y=1.08)
        fig.tight_layout(rect=(0, 0, 1, 0.92))
        fig.savefig(LMM_OUT / f'fig_lmm_trajectories_{m}.png', dpi=LMM_DPI)
        fig.savefig(LMM_OUT / f'fig_lmm_trajectories_{m}.pdf')
        plt.close(fig)

    # ---------------------------- FIGURE 2: slope differences ----------------------------
    ncol = 2 if len(lmm_ocs) > 1 else 1
    nrow = int(np.ceil(len(lmm_ocs) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(6.6 * ncol, 2.6 * nrow + 1.0), squeeze=False)
    axes = axes.ravel()
    letters = ['A', 'B', 'C', 'D', 'E', 'F']
    for i, (oc, oname, dr) in enumerate(lmm_ocs):
        ax = axes[i]
        rows = [lmm_res[(oc, m)] for m in LMM_METRICS if (oc, m) in lmm_res]
        ys = np.arange(len(rows))[::-1]
        fin = [r for r in rows if np.isfinite(r['diff'])]
        span = max([abs(r['diff_lo']) for r in fin] + [abs(r['diff_hi']) for r in fin] + [0.01])
        for y, r in zip(ys, rows):
            if not np.isfinite(r['diff']):
                ax.text(0, y, 'not estimable', ha='center', va='center', fontsize=7.5,
                        color='#A32D2D')
                continue
            pa = r.get('p_adj', np.nan)
            col = '#185FA5' if (np.isfinite(pa) and pa < LMM_ALPHA) else '#5F5E5A'
            ax.plot([r['diff_lo'], r['diff_hi']], [y, y], color=col, lw=1.5, solid_capstyle='butt')
            ax.plot(r['diff'], y, 'o', color=col, ms=5.5)
            ax.text(span * 1.28, y, f"{r['diff']:+.3f} ({r['diff_lo']:+.3f} to {r['diff_hi']:+.3f})"
                                    f"  {lmm_ptxt(r['p'])}{lmm_stars(pa)}",
                    va='center', fontsize=7.5, color=col)
        ax.axvline(0, color='#888780', ls='--', lw=0.8, zorder=0)
        ax.set_xlim(-span * 1.20, span * 3.4)
        ax.set_ylim(-0.7, len(rows) - 0.3)
        ax.set_yticks(ys)
        ax.set_yticklabels([f"{r['parameter']}\n({r['contrast']})" for r in rows], fontsize=8.5)
        ax.grid(axis='y', alpha=0)
        ax.set_xlabel(f'Difference in annual slope, exposure − reference ({oname} units/year)')
        ax.set_title(f'{oname}   (higher = {dr})', fontsize=9.5)
        ax.text(-0.10, 1.08, letters[i], transform=ax.transAxes, fontsize=13,
                fontweight='bold', va='bottom', ha='left')
    for ax in axes[len(lmm_ocs):]:
        ax.axis('off')
    fig.suptitle('Difference in annual rate of change, exposure vs reference kinetic group',
                 fontsize=12, y=1.0)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    fig.savefig(LMM_OUT / 'fig_lmm_slope_differences.png', dpi=LMM_DPI)
    fig.savefig(LMM_OUT / 'fig_lmm_slope_differences.pdf')
    plt.close(fig)

    # ---------------------------- summary ----------------------------
    print('\n'.join(lmm_md[:2 + len(lmm_tab)]))
    lmm_notes = [f"{r['outcome']}/{r['parameter']}: {r['note']}" for r in lmm_rows if r.get('note')]
    if lmm_notes:
        print('\nFit notes:')
        for c in lmm_notes:
            print('  ' + c)
    print('\nwritten to', LMM_OUT)

    return {'mode': mode, 'out': LMM_OUT, 'res': lmm_res, 'tab': lmm_tab}

## 1. Adjusted for phenoconversion

In [ ]:
run_adj = run_lmm_kinetics('adjust')

In [ ]:
lmm_n_per_year(run_adj)

## 2. Sensitivity analysis: phenoconverters excluded

In [ ]:
run_excl = run_lmm_kinetics('exclude')

In [ ]:
lmm_n_per_year(run_excl)